# NB50 — 63k Genis Veri: Hazirlama, Etiketleme, Sizinti Temizligi, EDA

Plan: `docs/PLAN_63K_ENTEGRASYON.md` ADIM 1.

Bu notebook `data/63k_genis/full_cravat_v3_63k.csv` (63.463 satir, 777 sutun, legacy
OpenCRAVAT semasi) dosyasini temiz, etiketli, sizintisiz parquet'e cevirir ve EDA yapar.
Sonraki tum notebook'lar (NB51+) bu notebook'un urettigi parquet + `src/columns_63k.py`
uzerinden calisir.

**Rejim:** Final test dagilimi varsayimi YOK (kullanici karari). Degerlendirme protokolu:
stratified hold-out + 5-fold CV + gen-holdout (GroupKFold by `base__hugo`).

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import SEED
from src import columns_63k as C63

np.random.seed(SEED)

RAW_CSV = os.path.join(PROJECT_ROOT, 'data', '63k_genis', 'full_cravat_v3_63k.csv')
PARQUET_DIR = os.path.join(PROJECT_ROOT, 'data', '63k_genis')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v31_63k_prep')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SEED:', SEED)
print('RAW_CSV exists:', os.path.exists(RAW_CSV), os.path.getsize(RAW_CSV) / 1e6, 'MB')

PROJECT_ROOT: /Users/tefe/teknofest_model/teknofest_model
SEED: 42
RAW_CSV exists: True 824.267802 MB


## Adim 1.1 — Parquet Donusumu (chunked)

824 MB CSV'yi tek seferde RAM'e almak yerine `chunksize=5000` ile parca parca okuyup
`object` sutunlari `category`'ye cevirerek yaziyoruz. Bir kez calisir; sonraki
notebook'lar dogrudan parquet okur.

In [2]:
# Cell 2: Chunked CSV -> Parquet donusumu (object -> category)
import pyarrow as pa
import pyarrow.parquet as pq

FULL_PARQUET = os.path.join(PARQUET_DIR, 'full_63k.parquet')
CHUNK_SIZE = 5000

if os.path.exists(FULL_PARQUET):
    print(f'{FULL_PARQUET} zaten var, donusum atlaniyor.')
else:
    # Object sutunlarin cardinality'sini ilk chunk'tan tahmin etmek yerine
    # tum chunk'lari tek bir DataFrame'de birlestirip sonra category'ye ceviriyoruz --
    # 63k satir icin bu hala bellekte tasinabilir (asil sisirme object dtype'in
    # Python-string overhead'inden geliyor, category donusumu bunu cozuyor).
    reader = pd.read_csv(RAW_CSV, chunksize=CHUNK_SIZE, low_memory=False)
    chunks = []
    for i, chunk in enumerate(reader):
        chunks.append(chunk)
        if (i + 1) % 5 == 0:
            print(f'  {(i+1) * CHUNK_SIZE} satir okundu...')
    df_full = pd.concat(chunks, ignore_index=True)
    del chunks
    print('Toplam satir:', len(df_full), 'sutun:', df_full.shape[1])

    # Chunked okuma, farkli chunk'larda ayni sutuna farkli dtype (int/str) atayabilir
    # (ornegin clinvar__dbsnp_id bazi chunk'larda tumu sayisal gorunup int, bazilarinda
    # NaN/string karisik oldugu icin object kalabilir) -- concat sonrasi object dtype
    # icindeki degerler hala karisik Python tipte olabilir. category'ye cevirmeden once
    # once str'e normalize ediyoruz ki pyarrow tek bir Arrow tipine cevirebilsin.
    obj_cols = df_full.select_dtypes(include='object').columns
    for c in obj_cols:
        df_full[c] = df_full[c].astype(str).replace('nan', np.nan).astype('category')

    df_full.to_parquet(FULL_PARQUET, engine='pyarrow', index=False)
    print(f'Yazildi: {FULL_PARQUET} ({os.path.getsize(FULL_PARQUET) / 1e6:.1f} MB)')
    del df_full

df_full = pd.read_parquet(FULL_PARQUET)
print('Parquet okundu:', df_full.shape)

/Users/tefe/teknofest_model/teknofest_model/data/63k_genis/full_63k.parquet zaten var, donusum atlaniyor.
Parquet okundu: (63463, 777)


## Adim 1.2 — Etiket Turetimi

`clinvar__sig` metninden `Label` (0/1) turetilir; `label_conf` guven agirligi
`clinvar__rev_stat`'tan hesaplanir; `Label_qualified` ek nitelikli (`|other` vb.)
etiketleri isaretler. Mantik `src/columns_63k.py::derive_label` icinde, bu notebook
sadece uygular ve dogrular.

In [3]:
# Cell 3: Etiket turetimi
df_full['Label'] = df_full[C63.LABEL_SOURCE_COL].astype(str).apply(C63.derive_label)
df_full['label_conf'] = df_full[C63.LABEL_REVSTAT_COL].astype(str).apply(C63.derive_label_confidence)
df_full['Label_qualified'] = df_full[C63.LABEL_SOURCE_COL].astype(str).apply(C63.is_qualified_label)
df_full['Label_excluded'] = df_full['Label'].isna()

n_excluded = df_full['Label_excluded'].sum()
print(f'Disillanan (belirsiz etiket) satir: {n_excluded} / {len(df_full)}')
print('Label dagilimi (tumu, disillananlar haric):')
print(df_full.loc[~df_full['Label_excluded'], 'Label'].value_counts())
print()
print('label_conf dagilimi (etiketli satirlarda):')
print(df_full.loc[~df_full['Label_excluded'], 'label_conf'].value_counts())
print()
print('Label_qualified sayisi:', df_full['Label_qualified'].sum())

assert n_excluded < 20, f'Beklenenden fazla disillama: {n_excluded} (beklenti ~11)'
df_labeled = df_full[~df_full['Label_excluded']].copy()
df_labeled['Label'] = df_labeled['Label'].astype(int)
print()
print('Etiketli veri:', df_labeled.shape)

Disillanan (belirsiz etiket) satir: 11 / 63463
Label dagilimi (tumu, disillananlar haric):
Label
0.0    39924
1.0    23528
Name: count, dtype: int64

label_conf dagilimi (etiketli satirlarda):
label_conf
0.9    58847
1.0     4604
0.6        1
Name: count, dtype: int64

Label_qualified sayisi: 97

Etiketli veri: (63452, 781)


## Adim 1.3 — Varyant Tipi Filtresi (missense-only ana set)

Ana modelleme seti `base__so == 'MIS'`. Non-missense satirlar ayri parquet'e
(`nonmis_63k.parquet`) yazilir — ADIM 4'te ek-veri ablasyonu icin saklanir.

In [4]:
# Cell 4: Missense filtresi + non-missense ayirma
is_missense = df_labeled[C63.VARIANT_TYPE_COL] == 'MIS'
df_missense = df_labeled[is_missense].copy()
df_nonmis = df_labeled[~is_missense].copy()

print('Missense (ana set):', df_missense.shape)
print('Non-missense (ek-veri ablasyonu icin ayri):', df_nonmis.shape)
print()
print('Missense Label dagilimi:')
print(df_missense['Label'].value_counts())
prevalence = df_missense['Label'].mean()
floor = C63.floor_f1(prevalence)
print(f'Prevalans (patho) = {prevalence:.4f}, floor-F1 = {floor:.4f}')

assert abs(prevalence - 0.373) < 0.01, f'Prevalans beklenenden sapti: {prevalence}'

NONMIS_PARQUET = os.path.join(PARQUET_DIR, 'nonmis_63k.parquet')
df_nonmis.to_parquet(NONMIS_PARQUET, engine='pyarrow', index=False)
print(f'Yazildi: {NONMIS_PARQUET}')

Missense (ana set): (60970, 781)
Non-missense (ek-veri ablasyonu icin ayri): (2482, 781)

Missense Label dagilimi:
Label
0    38248
1    22722
Name: count, dtype: int64
Prevalans (patho) = 0.3727, floor-F1 = 0.5430
Yazildi: /Users/tefe/teknofest_model/teknofest_model/data/63k_genis/nonmis_63k.parquet


## Adim 1.4 — Sizinti Temizligi (en kritik adim)

Drop edilenler: tum `clinvar__*` + `clinvar_acmg__*`, ID/serbest-metin/transkript
sutunlari. `LEAKY_META_PREDICTOR_SCORES`/`PREDS` **ayrilir ama drop edilmez**
(ADIM 3 A6 ablasyonu icin). `base__hugo` ve `base__chrom`/`base__pos` feature
olarak kullanilmaz, grup/split anahtari olarak saklanir.

In [ ]:
# Cell 5: Sizinti temizligi
# NB51 (sizinti denetimi) NB50'nin ilk sizinti listesinin eksik oldugunu buldu:
# rankscore/varity/vest ailesi LEAKY_META_PREDICTOR_SCORES'a, mupit__hugo/omim__omim_id/
# litvar_full__* ID_TEXT_TRANSCRIPT_COLS'a eklendi (bkz. src/columns_63k.py NB51 notu).
drop_cols = (set(C63.CLINVAR_LEAK_COLS) | set(C63.ID_TEXT_TRANSCRIPT_COLS)
             | set(C63.NB51_ADDITIONAL_ID_LEAK_COLS))
drop_cols &= set(df_missense.columns)  # veride olmayanlari sessizce yoksay

meta_score_cols = [c for c in C63.LEAKY_META_PREDICTOR_SCORES if c in df_missense.columns]
meta_rankscore_cols = [c for c in C63.LEAKY_META_PREDICTOR_RANKSCORES if c in df_missense.columns]
meta_pred_cols = [c for c in C63.LEAKY_META_PREDICTOR_PREDS if c in df_missense.columns]

missing_meta = set(C63.LEAKY_META_PREDICTOR_SCORES) - set(meta_score_cols)
if missing_meta:
    print('UYARI: CSV de bulunmayan meta-predictor sutunlari:', missing_meta)

before_n = df_missense.shape[1]
non_feature_present = [c for c in C63.NON_FEATURE_COLS if c in df_missense.columns]

df_clean = df_missense.drop(columns=list(drop_cols))
after_n = df_clean.shape[1]

print(f'Sizinti temizligi oncesi sutun: {before_n}')
print(f'Drop edilen (clinvar + id/text/transcript + NB51 ek ID sizintisi): {len(drop_cols)}')
print(f'Sizinti temizligi sonrasi sutun: {after_n}')
print(f'Meta-predictor skor sutunlari (AYRI tutuluyor, henuz drop edilmedi): {len(meta_score_cols)}')
print(f'Meta-predictor rankscore sutunlari (AYRI tutuluyor, henuz drop edilmedi): {len(meta_rankscore_cols)}')
print(f'Meta-predictor pred sutunlari (AYRI tutuluyor, henuz drop edilmedi): {len(meta_pred_cols)}')
print(f'Feature-disi (grup/split anahtari) sutunlar: {non_feature_present}')

# clinvar__* / clinvar_acmg__* hicbir sekilde kalmamali -- sert kontrol
remaining_clinvar = [c for c in df_clean.columns if c.startswith('clinvar__') or c.startswith('clinvar_acmg__')]
assert not remaining_clinvar, f'SIZINTI: clinvar sutunlari hala mevcut: {remaining_clinvar}'
print()
print('Dogrulandi: clinvar__* / clinvar_acmg__* sutunlari kalmadi.')

## Adim 1.5 — Constant + Duplicate Sutun Temizligi

`src/columns_real.py::get_constant_cols()` ve `get_duplicate_col_pairs()` sema-agnostik
yazilmis (sutun adi hardcode etmiyor) -- CLAUDE.md geregi dogrudan yeniden kullaniliyor.

In [6]:
# Cell 6: Constant + duplicate temizligi (sema-agnostik, columns_real'den yeniden kullanim)
from src.columns_real import get_constant_cols, get_duplicate_col_pairs

feature_candidate_cols = [c for c in df_clean.columns
                          if c not in C63.NON_FEATURE_COLS
                          and c not in ('Label', 'label_conf', 'Label_qualified', 'Label_excluded')]

constant_cols = get_constant_cols(df_clean[feature_candidate_cols])
print(f'Sabit sutun sayisi (nunique<=1): {len(constant_cols)}')

df_no_const = df_clean.drop(columns=constant_cols)
feature_candidate_cols_2 = [c for c in feature_candidate_cols if c not in constant_cols]

dup_pairs = get_duplicate_col_pairs(df_no_const[feature_candidate_cols_2])
dup_drop = sorted({b for (a, b) in dup_pairs})  # her ciftten ikinci sutunu drop et
print(f'Ozdes sutun cifti sayisi: {len(dup_pairs)} -> drop adayi sutun: {len(dup_drop)}')

df_final = df_no_const.drop(columns=dup_drop)
print()
print(f'Constant + duplicate temizligi sonrasi toplam sutun: {df_final.shape[1]}')
print(f'(Sizinti temizligi sonrasi {after_n} idi -> {len(constant_cols) + len(dup_drop)} daha drop edildi)')

Sabit sutun sayisi (nunique<=1): 70
Ozdes sutun cifti sayisi: 3 -> drop adayi sutun: 3

Constant + duplicate temizligi sonrasi toplam sutun: 538
(Sizinti temizligi sonrasi 611 idi -> 73 daha drop edildi)


## Adim 1.6 — Tip Ayristirma -> `src/columns_63k.py` icin girdi

Kalan sutunlar NUMERIC / CATEGORICAL / BINARY olarak siniflandirilir
(`classify_columns`, PRED_STRING_SUFFIXES ile bitenler kategorik sayilir).

In [7]:
# Cell 7: Tip ayristirma
exclude_for_typing = set(C63.NON_FEATURE_COLS) | {'Label', 'label_conf', 'Label_qualified', 'Label_excluded'}
col_types = C63.classify_columns(df_final, exclude_cols=exclude_for_typing)

print('NUMERIC:', len(col_types['numeric']))
print('CATEGORICAL:', len(col_types['categorical']))
print('BINARY:', len(col_types['binary']))
print('Toplam (feature):', sum(len(v) for v in col_types.values()))

pred_string_cols = C63.get_pred_string_cols(df_final.columns)
print()
print('*__pred / *__class (kategorik alt-kume):', len(pred_string_cols))

# meta-predictor / constant / duplicate listelerini sonuc olarak kaydet (Adim 3 icin referans)
import json as _json
meta_json = {
    'meta_score_cols': meta_score_cols,
    'meta_pred_cols': meta_pred_cols,
    'constant_cols': constant_cols,
    'duplicate_drop_cols': dup_drop,
    'numeric_cols': col_types['numeric'],
    'categorical_cols': col_types['categorical'],
    'binary_cols': col_types['binary'],
}
with open(os.path.join(RESULTS_DIR, 'nb50_column_lists.json'), 'w') as f:
    _json.dump(meta_json, f, indent=2, default=str)
print()
print('Sutun listeleri kaydedildi: results/v31_63k_prep/nb50_column_lists.json')

NUMERIC: 359
CATEGORICAL: 168
BINARY: 0
Toplam (feature): 527

*__pred / *__class (kategorik alt-kume): 6

Sutun listeleri kaydedildi: results/v31_63k_prep/nb50_column_lists.json


## Adim 1.7 — EDA

Sinif dengesi, eksiklik profili + MNAR/MCAR sorusu, gen bazli tablo, tek-degiskenli AUC,
yarisma verisiyle karsilastirma.

In [8]:
# Cell 8: EDA -- sinif dengesi + floor-F1 (kesin)
print('=== Sinif Dengesi (missense-only, sizinti-temiz) ===')
print(df_final['Label'].value_counts())
prevalence_final = df_final['Label'].mean()
floor_final = C63.floor_f1(prevalence_final)
print(f'Prevalans (patho) = {prevalence_final:.4f}')
print(f'FLOOR-F1 = {floor_final:.4f}  <-- her modelin gecmesi gereken alt sinir')
print(f'Majority-class accuracy baseline = {max(prevalence_final, 1 - prevalence_final):.4f}')

=== Sinif Dengesi (missense-only, sizinti-temiz) ===
Label
0    38248
1    22722
Name: count, dtype: int64
Prevalans (patho) = 0.3727
FLOOR-F1 = 0.5430  <-- her modelin gecmesi gereken alt sinir
Majority-class accuracy baseline = 0.6273


In [9]:
# Cell 9: EDA -- eksiklik profili + MNAR/MCAR sorusu (NB43 phi-korelasyon yontemi)
feature_cols_final = col_types['numeric'] + col_types['binary']  # kategorikler ayri ele alinir

missing_pct = df_final[feature_cols_final].isna().mean().sort_values(ascending=False)
print('En yuksek eksiklik oranli 20 sutun:')
print(missing_pct.head(20))
print()
print(f'Ortalama eksiklik (numeric+binary): {missing_pct.mean():.4f}')
print(f'>%50 eksik sutun sayisi: {(missing_pct > 0.5).sum()}')

# Eksiklik x Label korelasyonu (phi katsayisi = iki binary degisken arasi Pearson)
is_missing_df = df_final[feature_cols_final].isna().astype(int)
label_arr = df_final['Label'].values
phi_corr = is_missing_df.apply(lambda col: np.corrcoef(col.values, label_arr)[0, 1] if col.std() > 0 else 0.0)
phi_corr = phi_corr.sort_values(key=lambda s: s.abs(), ascending=False)

print()
print('Eksiklik x Label en guclu phi-korelasyonlu 15 sutun:')
print(phi_corr.head(15))

strong_mnar = (phi_corr.abs() > 0.1).sum()
print()
if strong_mnar > 10:
    print(f'SONUC: {strong_mnar} sutunda |phi|>0.1 -- eksiklik MNAR (Label ile iliskili), eski veriyle tutarli.')
else:
    print(f'SONUC: sadece {strong_mnar} sutunda |phi|>0.1 -- eksiklik buyuk olcude MCAR/random gorunuyor, eski veriden farkli.')

En yuksek eksiklik oranli 20 sutun:
hgdp__african_allele_freq                0.999869
hgdp__cs_asian_allele_freq               0.999869
aloft__tolerant                          0.999869
aloft__recessive                         0.999869
hgdp__native_american_allele_freq        0.999869
hgdp__oceanian_allele_freq               0.999869
hgdp__middle_eastern_allele_freq         0.999869
hgdp__european_allele_freq               0.999869
hgdp__east_asian_allele_freq             0.999869
aloft__dominant                          0.999869
cancer_genome_interpreter__responsive    0.999393
cancer_genome_interpreter__resistant     0.999393
cancer_genome_interpreter__other         0.999393
alfa_asian__asian_alt                    0.999131
alfa_asian__other_freq                   0.999131
alfa_asian__other_alt                    0.999131
alfa_asian__south_freq                   0.999131
alfa_asian__south_alt                    0.999131
alfa_asian__east_freq                    0.999131
alfa_asian__ea

In [10]:
# Cell 10: EDA -- gen bazli tablo (top-50, ezber riski)
gene_table = df_final.groupby(C63.GENE_GROUP_COL, observed=True)['Label'].agg(['count', 'sum', 'mean'])
gene_table.columns = ['n', 'n_pathogenic', 'prevalence']
gene_table['n_benign'] = gene_table['n'] - gene_table['n_pathogenic']
gene_table = gene_table.sort_values('n', ascending=False).head(50)
print(gene_table.to_string())

one_sided_genes = gene_table[(gene_table['prevalence'] > 0.95) | (gene_table['prevalence'] < 0.05)]
print()
print(f'Tek-yonlu genler (top-50 icinde, prevalans>0.95 veya <0.05): {len(one_sided_genes)}')
print(one_sided_genes[['n', 'prevalence']])

gene_table.to_csv(os.path.join(RESULTS_DIR, 'nb50_gene_table_top50.csv'))

              n  n_pathogenic  prevalence  n_benign
base__hugo                                         
FBN1        569           545    0.957821        24
LDLR        506           493    0.974308        13
PAH         405           404    0.997531         1
TTN         336             9    0.026786       327
BRCA1       335           151    0.450746       184
GCK         301           293    0.973422         8
BRCA2       289            81    0.280277       208
ABCA4       279           267    0.956989        12
SCN1A       274           253    0.923358        21
NF1         273           258    0.945055        15
TP53        272           199    0.731618        73
KMT2D       249            24    0.096386       225
GLA         238           234    0.983193         4
USH2A       222           135    0.608108        87
COL4A5      218           191    0.876147        27
ALPL        201           196    0.975124         5
MYH7        197           185    0.939086        12
COL1A2      

In [11]:
# Cell 11: EDA -- tek-degiskenli AUC (top-30), sizinti son-kontrol
from sklearn.metrics import roc_auc_score

univariate_auc = {}
for col in col_types['numeric'] + col_types['binary']:
    vals = df_final[col]
    mask = vals.notna()
    if mask.sum() < 100:
        continue
    v = vals[mask].astype(float)
    y = df_final.loc[mask, 'Label']
    if y.nunique() < 2 or v.nunique() < 2:
        continue
    try:
        auc = roc_auc_score(y, v)
        auc = max(auc, 1 - auc)  # yon farketmez, ayirt edicilik onemli
        univariate_auc[col] = auc
    except Exception:
        continue

auc_series = pd.Series(univariate_auc).sort_values(ascending=False)
print('Tek-degiskenli AUC top-30:')
print(auc_series.head(30))

red_flag_cols = auc_series[auc_series > 0.95]
print()
if len(red_flag_cols) > 0:
    print(f'!!! KIRMIZI BAYRAK: {len(red_flag_cols)} sutun tek basina AUC>0.95 -- NB51 sizinti denetiminde elle incelenmeli:')
    print(red_flag_cols)
else:
    print('Tek-degiskenli AUC>0.95 sutun yok -- ilk gozlemde belirgin sizinti isareti gorulmedi (NB51 kesin karari verecek).')

auc_series.to_csv(os.path.join(RESULTS_DIR, 'nb50_univariate_auc.csv'))

Tek-degiskenli AUC top-30:
ditto__score                          0.996841
metarnn__score                        0.996018
metarnn__rank_score                   0.996018
cardioboost__arrhythmias              0.995668
clinpred__rankscore                   0.994386
clinpred__score                       0.994386
bayesdel__bayesdel_addAF_rankscore    0.992926
bayesdel__bayesdel_addAF_score        0.992926
brca1_func_assay__score               0.980827
revel__score                          0.976320
revel__rankscore                      0.976320
bayesdel__bayesdel_noAF_score         0.974760
bayesdel__bayesdel_noAF_rankscore     0.974760
gmvp__score                           0.969198
gmvp__rank_score                      0.969198
vest__score                           0.968506
vest__pval                            0.968500
mistic__score                         0.966045
varity_r__varity_r                    0.964492
varity_r__varity_r_loo                0.960471
arrvars__lqt_penetrance          

In [12]:
# Cell 12: EDA -- yarisma verisiyle karsilastirma tablosu
comparison = pd.DataFrame({
    'Yarisma (MASTER, anonim)': {
        'n': 2931, 'prevalence_patho': 0.733, 'floor_f1': 0.846,
        'ortalama_eksiklik': 0.55, 'feature_sayisi': 351, 'yorumlanabilir': 'Hayir',
    },
    '63k (missense, legacy)': {
        'n': int(len(df_final)), 'prevalence_patho': round(float(prevalence_final), 4),
        'floor_f1': round(float(floor_final), 4),
        'ortalama_eksiklik': round(float(missing_pct.mean()), 4),
        'feature_sayisi': int(sum(len(v) for v in col_types.values())),
        'yorumlanabilir': 'Evet (SHAP)',
    },
}).T
print(comparison.to_string())
comparison.to_csv(os.path.join(RESULTS_DIR, 'nb50_vs_yarisma_comparison.csv'))

                              n prevalence_patho floor_f1 ortalama_eksiklik feature_sayisi yorumlanabilir
Yarisma (MASTER, anonim)   2931            0.733    0.846              0.55            351          Hayir
63k (missense, legacy)    60970           0.3727    0.543            0.3799            527    Evet (SHAP)


## Cikti: Temiz Parquet'ler + `src/columns_63k.py` Referans Listeleri

In [13]:
# Cell 13: Final parquet'leri yaz
MISSENSE_PARQUET = os.path.join(PARQUET_DIR, 'missense_63k.parquet')
df_final.to_parquet(MISSENSE_PARQUET, engine='pyarrow', index=False)
print(f'Yazildi: {MISSENSE_PARQUET} ({os.path.getsize(MISSENSE_PARQUET) / 1e6:.1f} MB), shape={df_final.shape}')

summary = {
    'n_total_labeled': int(len(df_labeled)),
    'n_missense': int(len(df_missense)),
    'n_nonmissense': int(len(df_nonmis)),
    'n_excluded_label': int(n_excluded),
    'n_cols_after_leak_cleanup': int(after_n),
    'n_constant_cols_dropped': int(len(constant_cols)),
    'n_duplicate_cols_dropped': int(len(dup_drop)),
    'n_cols_final': int(df_final.shape[1]),
    'prevalence_patho': round(float(prevalence_final), 4),
    'floor_f1': round(float(floor_final), 4),
    'n_meta_predictor_score_cols_retained_separately': len(meta_score_cols),
    'n_columns_auc_gt_095': int(len(red_flag_cols)),
    'mean_missing_pct': round(float(missing_pct.mean()), 4),
    'n_strong_mnar_cols_phi_gt_01': int(strong_mnar),
}
with open(os.path.join(RESULTS_DIR, 'nb50_summary.json'), 'w') as f:
    import json as _json
    _json.dump(summary, f, indent=2)

print()
print('=== NB50 OZET ===')
for k, v in summary.items():
    print(f'{k}: {v}')

Yazildi: /Users/tefe/teknofest_model/teknofest_model/data/63k_genis/missense_63k.parquet (73.9 MB), shape=(60970, 538)

=== NB50 OZET ===
n_total_labeled: 63452
n_missense: 60970
n_nonmissense: 2482
n_excluded_label: 11
n_cols_after_leak_cleanup: 611
n_constant_cols_dropped: 70
n_duplicate_cols_dropped: 3
n_cols_final: 538
prevalence_patho: 0.3727
floor_f1: 0.543
n_meta_predictor_score_cols_retained_separately: 27
n_columns_auc_gt_095: 30
mean_missing_pct: 0.3799
n_strong_mnar_cols_phi_gt_01: 124


## Bitis Kriteri Kontrolu

- [x] Temiz parquet (`missense_63k.parquet`, `nonmis_63k.parquet`, `full_63k.parquet`)
- [x] `src/columns_63k.py` (sizinti listeleri, etiket turetimi, tip ayristirma)
- [x] Sizinti sutunlari listesi belgelenmis (`results/v31_63k_prep/nb50_column_lists.json`)
- [x] Eksiklik MNAR/MCAR sorusu yanitlanmis (Cell 9 ciktisi)
- [ ] PDF rapor (`reports/nb50_63k_eda_report.pdf`) -- asagida uretiliyor

In [14]:
# Cell 14: Otomatik PDF rapor (fpdf2, NB10/NB13 pattern'i)
from fpdf import FPDF

class NB50Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.cell(0, 10, 'NB50 - 63k Genis Veri Hazirlama & EDA Raporu', ln=True, align='C')
        self.ln(2)

    def section(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 8, title, ln=True)
        self.set_font('Helvetica', '', 10)

    def kv_table(self, d):
        for k, v in d.items():
            self.cell(0, 6, f'{k}: {v}', ln=True)
        self.ln(2)

report = NB50Report()
report.add_page()

report.section('1. Veri Ozeti')
report.kv_table({
    'Kaynak dosya': 'data/63k_genis/full_cravat_v3_63k.csv',
    'Toplam satir': len(df_full),
    'Etiketlenen satir': len(df_labeled),
    'Disillanan (belirsiz etiket)': n_excluded,
    'Missense satir (ana set)': len(df_missense),
    'Non-missense satir (ayri parquet)': len(df_nonmis),
})

report.section('2. Sizinti Temizligi')
report.kv_table({
    'Sizinti temizligi oncesi sutun': before_n,
    'Drop edilen (clinvar + id/text/transcript)': len(drop_cols),
    'Sizinti temizligi sonrasi sutun': after_n,
    'Sabit sutun (drop)': len(constant_cols),
    'Ozdes sutun cifti (drop)': len(dup_drop),
    'Final sutun sayisi': df_final.shape[1],
    'Meta-predictor skor sutunlari (ayri tutuldu)': len(meta_score_cols),
})

report.section('3. Sinif Dengesi ve Floor-F1')
report.kv_table({
    'Prevalans (patho)': f'{prevalence_final:.4f}',
    'FLOOR-F1': f'{floor_final:.4f}',
    'Majority-class accuracy baseline': f'{max(prevalence_final, 1 - prevalence_final):.4f}',
})

report.section('4. Eksiklik Profili')
report.kv_table({
    'Ortalama eksiklik (numeric+binary)': f'{missing_pct.mean():.4f}',
    '>%50 eksik sutun sayisi': int((missing_pct > 0.5).sum()),
    'MNAR gucunde sutun sayisi (|phi|>0.1)': int(strong_mnar),
    'MNAR/MCAR sonucu': 'MNAR (Label ile iliskili)' if strong_mnar > 10 else 'MCAR\'a yakin',
})

report.section('5. Sizinti Riski Sinyali (NB51 icin on-bulgu)')
report.kv_table({
    'Tek-degiskenli AUC>0.95 sutun sayisi': len(red_flag_cols),
    'Not': 'Kesin karar NB51 sizinti denetiminde verilecek',
})

report.section('6. Yarisma Verisiyle Karsilastirma')
for idx, row in comparison.iterrows():
    report.set_font('Helvetica', 'B', 10)
    report.cell(0, 6, str(idx), ln=True)
    report.set_font('Helvetica', '', 9)
    for c in comparison.columns:
        report.cell(0, 5, f'  {c}: {row[c]}', ln=True)
    report.ln(1)

REPORT_PATH = os.path.join(REPORTS_DIR, 'nb50_63k_eda_report.pdf')
report.output(REPORT_PATH)
print(f'PDF rapor yazildi: {REPORT_PATH}')

PDF rapor yazildi: /Users/tefe/teknofest_model/teknofest_model/reports/nb50_63k_eda_report.pdf


## Sonraki Adim

**ADIM 2 — NB51 (`notebooks/51_63k_leakage_audit.ipynb`):** Hizli sinyal testi (LGBM,
CV F1 > 0.97 kirmizi bayrak), tek-sutun AUC taramasi (>0.95 elle incele), meta-predictor
ablasyonu (A6 on-hazirligi), gen-ezberi testi (GroupKFold vs StratifiedKFold),
duplicate satir kontrolu. Bu notebook'un ciktilari (`missense_63k.parquet`,
`nb50_column_lists.json`, tek-degiskenli AUC listesi) NB51'in girdisidir.